In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import peak_local_max
from scipy import ndimage
from skimage.segmentation import watershed
import math
import json
import glob
import random
import json
import os
import shutil
import random



#Pending 006 , 007, 008
all_images = glob.glob(r'Clenead_Images\PU122\XY_clean\*.jpg')
batch_size = 100
num_batches = math.ceil(len(all_images) / batch_size)
csv_filename = "Results/V3.csv"
print("Total de imágenes encontradas:", len(all_images))
print(all_images)

Total de imágenes encontradas: 86
['Clenead_Images\\PU122\\XY_clean\\xy008.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy009.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy010.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy011.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy012.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy013.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy014.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy015.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy016.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy017.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy018.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy019.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy020.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy021.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy022.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy023.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy024.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy025.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy026.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy027.jpg', 'Clenead_Images\\PU122\\XY_clean\\xy028.jpg',

In [12]:
def get_color_by_increment(area, increment=150):
    # Determine the class (range) of the area
    class_id = area // increment  # Integer division
    random.seed(class_id)  # Ensure consistent color for the same class
    return (
        random.randint(0, 255), 
        random.randint(0, 255), 
        random.randint(0, 255)
    ) 
def classify_by_size(area,increase=150):
    class_id = area // increase
    return class_id

In [13]:

# Listas para almacenar las anotaciones COCO
coco_annotations = []
coco_images = []

# Lista de categorías
coco_categories = [
    {
        "id": 1,
        "name": "burbuja",
        "supercategory": "objeto"
    }
]

annotation_id = 1  # Contador para las anotaciones
image_id = 1  # Contador para las imágenes

# Ruta para guardar las máscaras de segmentación
mask_output_dir = 'masks/'
os.makedirs(mask_output_dir, exist_ok=True)

for i in range(num_batches):
    batch_images = all_images[i * batch_size : (i + 1) * batch_size]

    for image_path in batch_images:
        # Definir umbrales y parámetros iniciales
        max_area_threshold = 10500
        min_area_threshold = 10
        kernel_size = 3
        morph_iterations = 2
        min_distance_peak = 8

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        image_name = os.path.splitext(os.path.basename(image_path))[0]
        height_img, width_img = image.shape[:2]

        # Agregar información de la imagen al formato COCO
        coco_images.append({
            "id": image_id,
            "width": width_img,
            "height": height_img,
            "file_name": os.path.basename(image_path)
        })

        # 1. Suavizar la imagen
        blur = cv2.GaussianBlur(image, (3, 3), 0)

        # 2. Umbral Otsu
        _, binary = cv2.threshold(blur, 80, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        binary = cv2.bitwise_not(binary)

        # 3. Limpiar por morfología
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        binary_opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=morph_iterations)

        # 4. Transformada de distancia
        dist_transform = cv2.distanceTransform(binary_opened, cv2.DIST_L2, 5)

        # 5. Picos locales
        coordinates = peak_local_max(dist_transform, min_distance=min_distance_peak, threshold_abs=0.5)
        local_max = np.zeros(dist_transform.shape, dtype=bool)
        local_max[tuple(coordinates.T)] = True

        # Etiquetar burbujas
        markers, _ = ndimage.label(local_max)

        # 6. Identificación de burbujas
        labels = watershed(-dist_transform, markers, mask=binary_opened)

        num_labels = labels.max()
        valid_count = 0
        pre_value = 0

        # Proceso para calcular porcentaje no pintado
        total_pixels = binary_opened.size
        painted_pixels = np.count_nonzero(binary_opened)
        unpainted_pixels = total_pixels - painted_pixels
        unpainted_percentage = (unpainted_pixels / total_pixels) * 100
        unpainted_percentage_adjusted = max(0, unpainted_percentage - 15.35)

        for lbl in range(1, num_labels + 1):
            mask = (labels == lbl).astype(np.uint8)
            area_px = cv2.countNonZero(mask)
            x, y, w, h = cv2.boundingRect(mask)
            aspect_ratio = w / h if h > 0 else 0

            # Filtro preliminar de burbujas válidas
            if 0.79 <= aspect_ratio <= 2.5 and min_area_threshold <= area_px <= max_area_threshold:
                pre_value += 1

        # Condiciones según el porcentaje no pintado ajustado
        if unpainted_percentage_adjusted > 25 and pre_value < 500:
            print(f"Unpainted percentage ({unpainted_percentage_adjusted}%) is too high. Recalculating...")
            # Recalcular con ajustes
            # 1. Suavizar la imagen para reducir ruido
            blur = cv2.GaussianBlur(image, (5, 5), 0)
            # 2. Binarizar usando umbral de Otsu
            _, binary = cv2.threshold(blur, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            binary = cv2.bitwise_not(binary)
            # 3. Operaciones morfológicas para limpiar pequeñas manchas y separar un poco las burbujas
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
            binary_opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
            # 4. Transformada de distancia
            dist_transform = cv2.distanceTransform(binary_opened, cv2.DIST_L2, 5)
            # 5. Encontrar los picos locales en la transformada de distancia
            coordinates = peak_local_max(dist_transform, min_distance=10, threshold_abs=0.5)
            local_max = np.zeros(dist_transform.shape, dtype=bool)
            local_max[tuple(coordinates.T)] = True

            # Etiquetar los picos para usarlos como marcadores en watershed
            markers, _ = ndimage.label(local_max)

            # 6. Aplicar watershed con la transformada de distancia negativa (para encontrar valles)
            labels = watershed(-dist_transform, markers, mask=binary_opened)
        else:
            print(f"Unpainted percentage ({unpainted_percentage_adjusted}%) is acceptable. Proceeding with results...")

        # Proceso para identificar y guardar burbujas
        for lbl in range(1, labels.max() + 1):
            mask = (labels == lbl).astype(np.uint8)
            area_px = cv2.countNonZero(mask)
            if area_px == 0:
                continue  # Evitar divisiones por cero
            x, y, w, h = cv2.boundingRect(mask)
            aspect_ratio = w / h if h > 0 else 0

            if 0.7 <= aspect_ratio < 2.0 and min_area_threshold <= area_px <= max_area_threshold:
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if len(contours) > 0:
                    segmentation = []
                    for contour in contours:
                        if len(contour) >= 3:
                            # Convertir los puntos del contorno a una lista plana
                            contour = contour.flatten().tolist()
                            segmentation.append(contour)

                    # Crear anotación en formato COCO
                    coco_annotation = {
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": 1,  # Clase burbuja
                        "segmentation": segmentation,
                        "area": float(area_px),
                        "bbox": [float(x), float(y), float(w), float(h)],
                        "iscrowd": 0
                    }
                    coco_annotations.append(coco_annotation)
                    annotation_id += 1

                    # Guardar la máscara de la burbuja individual
                    mask_filename = f"{image_name}_mask_{annotation_id}.png"
                    mask_path = os.path.join(mask_output_dir, mask_filename)
                    cv2.imwrite(mask_path, mask * 255)  # Multiplicar por 255 para convertir la máscara a imagen binaria

        # Incrementar el ID de imagen después de procesar todas las burbujas de una imagen
        image_id += 1

# Al finalizar el procesamiento de todas las imágenes, generar el archivo JSON en formato COCO
coco_dataset = {
    "info": {
        "description": "Dataset de burbujas",
        "url": "",
        "version": "1.0",
        "year": 2023,
        "contributor": "",
        "date_created": ""
    },
    "licenses": [
        {
            "id": 1,
            "name": "Attribution-NonCommercial-ShareAlike License",
            "url": "http://creativecommons.org/licenses/by-nc-sa/2.0/"
        }
    ],
    "images": coco_images,
    "annotations": coco_annotations,
    "categories": coco_categories
}

# Guardar el archivo JSON
with open('annotations.json', 'w') as json_file:
    json.dump(coco_dataset, json_file)

Unpainted percentage (17.323369323399895%) is acceptable. Proceeding with results...
Unpainted percentage (15.92029738198933%) is acceptable. Proceeding with results...
Unpainted percentage (16.426978485597033%) is acceptable. Proceeding with results...
Unpainted percentage (16.547091497036263%) is acceptable. Proceeding with results...
Unpainted percentage (16.08614685642184%) is acceptable. Proceeding with results...
Unpainted percentage (17.043032568599216%) is acceptable. Proceeding with results...
Unpainted percentage (16.995834876756305%) is acceptable. Proceeding with results...
Unpainted percentage (16.602910439091964%) is acceptable. Proceeding with results...
Unpainted percentage (17.184041152588094%) is acceptable. Proceeding with results...
Unpainted percentage (15.989486568019109%) is acceptable. Proceeding with results...
Unpainted percentage (16.559511942258077%) is acceptable. Proceeding with results...
Unpainted percentage (15.80975541951515%) is acceptable. Proceeding

In [16]:


# Rutas
dataset_imagen = r'Clenead_Images\PU122\XY_clean'
dataset_json = r'Clenead_Images\PU122\XY_clean'
images_dir = os.path.join(dataset_imagen, 'images')
train_images_dir = os.path.join(dataset_imagen, 'train', 'images')
val_images_dir = os.path.join(dataset_imagen, 'val', 'images')
train_annotations_dir = os.path.join(dataset_imagen, 'train', 'annotations')
val_annotations_dir = os.path.join(dataset_imagen, 'val', 'annotations')

# Crear carpetas si no existen
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(train_annotations_dir, exist_ok=True)
os.makedirs(val_annotations_dir, exist_ok=True)

# Proporción de entrenamiento y validación
train_ratio = 0.8  # 80% entrenamiento, 20% validación

# Cargar el archivo JSON original
with open(os.path.join(dataset_json, 'annotations.json'), 'r') as f:
    coco_data = json.load(f)


In [17]:

# Obtener todas las imágenes
images = coco_data['images']

# Barajar aleatoriamente las imágenes
random.shuffle(images)

# Calcular el número de imágenes para entrenamiento
train_count = int(len(images) * train_ratio)

# Dividir las imágenes
train_images = images[:train_count]
val_images = images[train_count:]

# Crear conjuntos de IDs de imágenes
train_image_ids = set([img['id'] for img in train_images])
val_image_ids = set([img['id'] for img in val_images])

# Dividir las anotaciones
annotations = coco_data['annotations']
train_annotations = [ann for ann in annotations if ann['image_id'] in train_image_ids]
val_annotations = [ann for ann in annotations if ann['image_id'] in val_image_ids]

# Crear los diccionarios COCO para entrenamiento y validación
train_coco = {
    'info': coco_data.get('info', {}),
    'licenses': coco_data.get('licenses', []),
    'images': train_images,
    'annotations': train_annotations,
    'categories': coco_data.get('categories', [])
}

val_coco = {
    'info': coco_data.get('info', {}),
    'licenses': coco_data.get('licenses', []),
    'images': val_images,
    'annotations': val_annotations,
    'categories': coco_data.get('categories', [])
}


In [18]:

# Guardar los nuevos archivos JSON
with open(os.path.join(train_annotations_dir, 'instances_train.json'), 'w') as f:
    json.dump(train_coco, f)

with open(os.path.join(val_annotations_dir, 'instances_val.json'), 'w') as f:
    json.dump(val_coco, f)

# Mover las imágenes a sus respectivas carpetas
for img in train_images:
    src = os.path.join(images_dir, img['file_name'])
    dst = os.path.join(train_images_dir, img['file_name'])
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print(f"Image not found: {src}")

for img in val_images:
    src = os.path.join(images_dir, img['file_name'])
    dst = os.path.join(val_images_dir, img['file_name'])
    if os.path.exists(src):
        shutil.copy(src, dst)